In [19]:
import numpy as np  # NumPy를 불러옵니다
import scipy as sp  # SciPy를 불러옵니다
import matplotlib.pyplot as plt  # 그래프 모듈을 불러옵니다
import pandas as pd  # pandas를 불러옵니다
import sympy as sy  # SymPy를 불러옵니다
sy.init_printing()  # 수식을 보기 좋게 표시합니다

In [3]:
from IPython.core.interactiveshell import InteractiveShell  # 대화형 셸 설정 도구를 불러옵니다
InteractiveShell.ast_node_interactivity = "all"  # 셀의 모든 표현식 결과를 표시합니다

# <font face="gotham" color="purple"> LU 분해</font>

LU 분해는 계산 효율을 높이기 위한 방법으로, 주로 연립일차방정식을 풀 때 사용합니다. LU 분해 자체의 부동소수점 연산 횟수는 가우스-조르당 소거법보다 많을 수 있습니다.

하지만 같은 계수행렬 $A$에 대해 여러 개의 우변 벡터로 $Ax=b$를 반복해서 풀 때 특히 유용합니다.

예를 들어 여러 벡터 $\{b_i,\ i \in \mathbb Z\}$를 차례로 대입해야 한다고 합시다.
$$
Ax=b_1,\quad Ax=b_2,\quad Ax=b_3,\quad Ax=b_4, \quad...
$$
LU 분해는 $A$를 한 번만 분해해 재사용할 수 있지만, 가우스-조르당 소거법은 각 확대행렬 $[A|b_i]$마다 소거 과정을 다시 수행해야 합니다.

## <font face="gotham" color="purple"> LU 분해 알고리즘</font>

행렬 $A$를 각각 _하삼각행렬_과 _상삼각행렬_을 뜻하는 $L$과 $U$의 곱으로 분해하는 것이 목표입니다. 여기서는 $L$의 주대각선 원소를 모두 1로 둡니다.
$$
A = LU
$$
예를 들면 다음과 같습니다.
$$
A=\underbrace{\left[\begin{array}{cccc}
1 & 0 & 0 & 0 \\
* & 1 & 0 & 0 \\
* & * & 1 & 0 \\
* & * & * & 1
\end{array}\right]}_{L}\underbrace{\left[\begin{array}{ccccc}
\blacksquare & * & * & * & * \\
0 & \blacksquare & * & * & * \\
0 & 0 & 0 & \blacksquare & * \\
0 & 0 & 0 & 0 & 0
\end{array}\right]}_{U}
$$

이 분해가 가능한 이유는 일련의 행 연산으로 $A$를 상삼각행렬로 바꿀 수 있기 때문입니다.

$$E_p...E_2E_1  A = U$$

식을 정리하면 다음과 같습니다.

$$A = (E_p...E_2E_1)^{-1}U = LU$$

따라서 $L$은 다음과 같습니다.

$$L = (E_p...E_2E_1)^{-1}$$

다음 행렬을 손으로 직접 분해해 보겠습니다.

$$
A =
\begin{bmatrix}
9 & 3 & 6\\3 & 4 & 6\\0 & 8 & 8\\
\end{bmatrix}
$$

$$
\underbrace{
\begin{bmatrix}
1 & 0 & 0\\-\frac{1}{3} & 1 & 0\\0 & 0 & 1\\
\end{bmatrix}}_{E_1}
\underbrace{
\begin{bmatrix}
9 & 3 & 6\\3 & 4 & 6\\0 & 8 & 8\\ 
\end{bmatrix}}_{A}
=
\underbrace{
\begin{bmatrix}
9 & 3 & 6\\0 & 3 & 4\\0 & 8 & 8\\ 
\end{bmatrix}}_{E_1A}
$$

$$
\underbrace{
\begin{bmatrix}
1 & 0 & 0\\0 &1  & 0\\0 & -\frac{8}{3} & 1\\
\end{bmatrix}}_{E_2}
\underbrace{
\begin{bmatrix}
9 & 3 & 6\\0 & 3 & 4\\0 & 8 & 8\\ 
\end{bmatrix}}_{E_1A}
=
\underbrace{
\begin{bmatrix}
9 & 3 & 6\\0 & 3 & 4\\0 & 0 & -\frac{8}{3}\\ 
\end{bmatrix}}_{E_2E_1A=U}
$$

$$
L^{-1} = E_2E_1 = 
\underbrace{
\begin{bmatrix}
1 & 0 & 0\\0 &1  & 0\\0 & -\frac{8}{3} & 1\\
\end{bmatrix}}_{E_2}
\underbrace{
\begin{bmatrix}
1 & 0 & 0\\-\frac{1}{3} & 1 & 0\\0 & 0 & 1\\
\end{bmatrix}}_{E_1}
=
\begin{bmatrix}
1 & 0 & 0\\-\frac{1}{3} & 1 & 0\\\frac{8}{9} & -\frac{8}{3} & 1\\
\end{bmatrix}
$$

또는 다음 관계를 이용해 $E_1^{-1}$과 $E_2^{-1}$을 직접 계산할 수도 있습니다.
$$
L = E_1^{-1}E_2^{-1}
$$


$E_1$과 $E_2$에 단위행렬을 붙여 확대행렬을 만듭니다.
$$
[E_1| I]=\left[\begin{array}{ccc|ccc}  
1 & 0 & 0 & 1 & 0 & 0\\
-\frac{1}{3} & 1 & 0 & 0  & 1 & 0\\
0 & 0 & 1 & 0 & 0 & 1
\end{array}\right]
\sim
\left[\begin{array}{ccc|ccc}
1 &0 &0 &1 & 0 & 0\\
0& 1& 0& \frac{1}{3} & 1 & 0\\
0 & 0 & 1 &0 & 0 & 1
\end{array}\right]
=[I|E_1^{-1}]
$$

$$
[E_2| I]=
\left[\begin{array}{ccc|ccc}  
1 & 0 & 0 & 1 & 0 & 0\\
0 & 1 & 0 & 0  & 1 & 0\\
0 & -\frac{8}{3} & 1 & 0 & 0 & 1\\
\end{array}
\right]
\sim
\left[\begin{array}{ccc|ccc}  
1 & 0 & 0 & 1 & 0 & 0\\
0 & 1 & 0 & 0  & 1 & 0\\
0 & 0 & 1 & 0 &\frac{8}{3}& 1\\
\end{array}
\right]
=[I|E_2^{-1}]
$$

그 결과 $E_1$과 $E_2$의 역행렬은 다음과 같습니다.
$$
E_1^{-1} = \left[\begin{array}{ccc} 1& 0& 0 \\ \frac{1}{3}& 1& 0\\ 0& 0 &1  \end{array}\right]\\
E_2^{-1} = \left[\begin{array}{ccc} 1& 0& 0 \\ 1& 1& 0\\ 0& \frac{8}{3} &1  \end{array}\right]
$$

$$
L = E_1^{-1}E_2^{-1}= 
\left[\begin{array}{ccc} 1& 0& 0 \\ \frac{1}{3}& 1& 0\\ 0& 0 &1  \end{array}\right]
\left[\begin{array}{ccc} 1& 0& 0 \\ 1& 1& 0\\ 0& \frac{8}{3} &1 \end{array}\right]
=
\left[\begin{array}{ccc} 1& 0& 0 \\ \frac{4}{3}&1& 0\\ 0& \frac{8}{3} &1 \end{array}\right]
$$

최종 LU 분해 결과는 다음과 같습니다.
$$
A = LU =
\underbrace{
\left[\begin{array}{ccc} 1& 0& 0 \\ \frac{4}{3}&1& 0\\ 0& \frac{8}{3} &1 \end{array}\right]}_{L}
\underbrace{
\begin{bmatrix}
9 & 3 & 6\\0 & 3 & 4\\0 & 0 & -\frac{8}{3}\\ 
\end{bmatrix}}_{E_2E_1A=U}
$$

SciPy의 계산 결과와 비교해 보겠습니다.

In [3]:
A = np.array([[9, 3, 6], [3, 4, 6], [0, 8, 8]]); A  # LU 분해할 행렬을 만들고 표시합니다

array([[9, 3, 6],
       [3, 4, 6],
       [0, 8, 8]])

In [4]:
P, L, U = sp.linalg.lu(A)  # 행렬을 치환행렬, 하삼각행렬, 상삼각행렬로 분해합니다
P  # 치환행렬을 표시합니다
L  # 하삼각행렬을 표시합니다
U  # 상삼각행렬을 표시합니다

array([[1., 0., 0.],
       [0., 0., 1.],
       [0., 1., 0.]])

array([[1.        , 0.        , 0.        ],
       [0.        , 1.        , 0.        ],
       [0.33333333, 0.375     , 1.        ]])

array([[9., 3., 6.],
       [0., 8., 8.],
       [0., 0., 1.]])

In [5]:
P@L@U  # 세 행렬을 곱해 원래 행렬 A가 복원되는지 확인합니다

array([[9., 3., 6.],
       [3., 4., 6.],
       [0., 8., 8.]])

SciPy의 \`lu\` 함수는 $L$과 $U$뿐 아니라 _치환행렬_ $P$도 반환합니다. 이론적으로는 행 교환 없이 $A$를 $U$로 만들 수 있는 경우도 있지만, 피벗이 0이거나 수치적으로 불안정한 경우에는 먼저 행을 바꾸어야 분해를 안정적으로 수행할 수 있습니다.

따라서 SciPy는 행 교환을 포함한 다음 형태의 분해를 사용합니다.
$$
A  = PLU
$$

치환행렬은 직교행렬이므로 일반적으로 $P^{-1}=P^T$입니다. 아래처럼 한 번의 행 교환을 나타내는 치환행렬은 대칭이어서 $P=P^{-1}$도 성립합니다. 확대행렬로 확인하면 행 교환 기본행렬의 역행렬이 자기 자신임을 알 수 있습니다.

$$
[P| I]=\left[\begin{array}{ccc|ccc}
1 & 0 & 0 & 1 & 0 & 0\\
0 & 0 & 1 & 0 & 1 & 0\\
0 & 1 & 0 & 0 & 0 & 1
\end{array}\right]
\sim
\left[\begin{array}{ccc|ccc}
1 & 0 & 0 & 1 & 0 & 0\\
0 & 1 & 0 & 0 & 0 & 1\\
0 & 0 & 1 & 0 & 1 & 0
\end{array}\right]=[I| P^{-1}]  =[P^{-1}|I]
$$

이 관점에서는 실제로 다음 관계를 분해한다고 볼 수 있습니다.

$$
PA = LU
$$

$$
\begin{align}
E_2E_1E_0A &=
\underbrace{
\begin{bmatrix}
1 & 0& 0\\0 & 1 & 0\\0 & -\frac{3}{8} & 1\\
\end{bmatrix}}_{E_2}
\underbrace{
\begin{bmatrix}
1 & 0& 0\\0 & 1 & 0\\-\frac{1}{3} & 0 & 1\\
\end{bmatrix}}_{E_1}
\underbrace{
\begin{bmatrix}
1 & 0& 0\\0 & 0 & 1\\0 & 1 & 0\\
\end{bmatrix}}_{E_0}
\begin{bmatrix}
9 & 3 & 6\\3 & 4 & 6\\0 & 8 & 8\\
\end{bmatrix}\\
E_2E_1E_0A &=
\underbrace{
\begin{bmatrix}
1 & 0& 0\\0 & 1 & 0\\0 & -\frac{3}{8} & 1\\
\end{bmatrix}}_{E_2}
\underbrace{
\begin{bmatrix}
1 & 0& 0\\0 & 1 & 0\\-\frac{1}{3} & 0 & 1\\
\end{bmatrix}}_{E_1}
\underbrace{
\begin{bmatrix}
9& 3& 6\\0&8 &8\\ 3& 4& 6
\end{bmatrix}}_{E_0A}\\
E_2E_1E_0A &=
\underbrace{
\begin{bmatrix}
1 & 0& 0\\0 & 1 & 0\\0 & -\frac{3}{8} & 1\\
\end{bmatrix}}_{E_2}
\underbrace{
\begin{bmatrix}
9 &3 &6\\0 &8& 8 \\0& 3& 4
\end{bmatrix}}_{E_1E_0A}\\
E_2E_1E_0A &=
\underbrace{
\begin{bmatrix}
9 &3 &6\\0 &8& 8 \\0& 0& -1
\end{bmatrix}}_{E_2E_1E_0A}=U
\end{align}
$$
식을 정리하면 $P$와 $L$을 다음처럼 확인할 수 있습니다.
$$
A = \underbrace{E_0^{-1}}_{P} \underbrace{(E_1^{-1}E_2^{-1})}_{L}U
$$

# <font face="gotham" color="purple"> LU 분해로 연립일차방정식 풀기</font>

다음 연립일차방정식을 풀어 봅시다.
$$
\begin{align}
3x_1-7x_2 -2x_3+2x_4&=-9\\
-3x_1+5x_2 +x_3 &=5\\
6x_1-4x_2 -5x_4&=7\\
-9x_1+5x_2 -5x_3+12x_4&=11\\
\end{align}
$$
행렬 형태로 쓰면 다음과 같습니다.
$$
\underbrace{\left[\begin{array}{rrrr}
3 & -7 & -2 & 2 \\
-3 & 5 & 1 & 0 \\
6 & -4 & 0 & -5 \\
-9 & 5 & -5 & 12
\end{array}\right]}_{A}
\left[\begin{array}{r}
x_1 \\
x_2 \\
x_3 \\
x_4
\end{array}\right]
=
\underbrace{\left[\begin{array}{r}
-9 \\
5 \\
7 \\
11
\end{array}\right]}_{b}
$$
계수행렬 $A$를 $LU$ 분해합니다.

$$\underbrace{\left[\begin{array}{rrrr}
3 & -7 & -2 & 2 \\
-3 & 5 & 1 & 0 \\
6 & -4 & 0 & -5 \\
-9 & 5 & -5 & 12
\end{array}\right]}_{A}
=\underbrace{
\left[\begin{array}{rrrr}
1 & 0 & 0 & 0 \\
-1 & 1 & 0 & 0 \\
2 & -5 & 1 & 0 \\
-3 & 8 & 3 & 1
\end{array}\right]}_{L}\underbrace{\left[\begin{array}{rrrr}
3 & -7 & -2 & 2 \\
0 & -2 & -1 & 2 \\
0 & 0 & -1 & 1 \\
0 & 0 & 0 & -1
\end{array}\right]}_{U}$$

$A$를 $LU$로 바꾸면 $L(Ux)=b$가 됩니다. 중간 벡터 $y=Ux$를 두고 아래 두 식을 차례로 풉니다.

$$
Ly = b\\
Ux = y
$$

먼저 확대행렬 $[L|b]$를 구성합니다.

$$
\underbrace{
\left[\begin{array}{rrrr}
1 & 0 & 0 & 0 \\
-1 & 1 & 0 & 0 \\
2 & -5 & 1 & 0 \\
-3 & 8 & 3 & 1
\end{array}\right]}_{L}
\underbrace{\left[\begin{array}{r}
y_1 \\
y_2 \\
y_3 \\
y_4
\end{array}\right]}_{y}
=
\underbrace{\left[\begin{array}{r}
-9 \\
5 \\
7 \\
11
\end{array}\right]}_{b}
$$

$$
\left[\begin{array}{rrrr|r}
1 & 0 & 0 & 0 & -9 \\
-1 & 1 & 0 & 0 & 5\\
2 & -5 & 1 & 0 & 7\\
-3 & 8 & 3 & 1 & 11
\end{array}\right]\sim
\left[\begin{array}{rrrr|r}
1 & 0 & 0 & 0 & -9 \\
0 & 1 & 0 & 0 & -4\\
0 & 0 & 1 & 0 & 5\\
0 & 0 & 0 & 1 & 1
\end{array}\right]
$$

$$
\left[\begin{array}{r}
y_1 \\
y_2 \\
y_3 \\
y_4
\end{array}\right]=
\left[\begin{array}{r}
-9 \\
-4 \\
5 \\
1
\end{array}\right]
$$

이제 $Ux=y$를 풀겠습니다. NumPy로 계산하면 다음과 같습니다.

In [6]:
U = np.array([[3, -7, -2, 2], [0, -2, -1, 2], [0, 0, -1, 1], [0, 0, 0,-1]])  # 상삼각행렬 U를 만듭니다
y = np.array([-9, -4, 5, 1])  # 전진대입으로 구한 중간 벡터 y를 만듭니다
np.linalg.solve(U, y)  # 후진대입에 해당하는 선형계 Ux=y를 풉니다

array([ 3.,  4., -6., -1.])

앞의 과정이 맞다면 이 값이 원래 연립방정식의 해입니다. `np.linalg.solve()`로 직접 푼 결과와 비교해 보겠습니다.

In [7]:
A = np.array([[3, -7, -2, 2], [-3, 5, 1, 0], [6, -4, 0, -5], [-9, 5, -5, 12]])  # 원래 연립방정식의 계수행렬을 만듭니다
b = np.array([-9, 5, 7, 11])  # 원래 연립방정식의 우변 벡터를 만듭니다
np.linalg.solve(A, b)  # 원래 선형계를 직접 풀어 결과를 검산합니다

array([ 3.,  4., -6., -1.])

두 결과가 같으므로 LU 분해로 올바른 해를 구했음을 확인할 수 있습니다.

# <font face="gotham" color="purple"> 촐레스키 분해</font>

촐레스키 분해는 행렬의 제곱근을 구하는 것과 비슷한 역할을 합니다. 대표적으로 공분산행렬을 분해할 때 사용하며, 이때 $L$은 독립적인 성분에 원하는 상관 구조를 부여하는 변환으로 작용합니다.
$$
\Sigma = L L^T
$$

서로 상관되지 않은 확률변수로 이루어진 벡터 $\mathbf{z}$를 생각해 봅시다. $\mathbf{z}$의 각 성분은 평균이 0이고 분산이 1인 표준정규 확률변수입니다. 즉 $\mathbf{z} \sim \mathcal{N}(0,I)$이며 $I$는 단위행렬입니다.
$$
\mathbf{z}=\left(\begin{array}{c}
z_1 \\
z_2 \\
\vdots \\
z_n
\end{array}\right)
$$

$z$에 $L$을 곱하면 원하는 상관 구조를 가진 벡터 $x$로 변환할 수 있습니다.
$$
x = Lz
$$

이를 확인하기 위해 변환된 벡터 \(x\)의 공분산행렬을 계산해 봅시다.
$$
\text{Cov}(x) = \text{Cov}(Lz) = L \text{Cov}(z) L^T = L I L^T = L L^T = \Sigma
$$
따라서 $\text{Cov}(x)$가 목표로 한 공분산 구조 $\Sigma$와 같음을 알 수 있습니다.

수치 예제로 확인해 보겠습니다.

In [32]:
# 대칭 표본을 사용해 표본 평균의 변동을 줄입니다
n = 1000  # 기본 표본 개수를 지정합니다
z = np.random.normal(size=(1000, 3))  # 독립 표준정규 표본을 생성합니다
z2 = -z  # 각 표본과 부호가 반대인 대칭 표본을 만듭니다
z = np.concatenate((z, z2), axis=0)  # 원본과 대칭 표본을 세로로 합칩니다

cov_matrix = np.array([[1, 0.8, 0.7], [0.8, 1, 0.8], [0.7, 0.8, 1]])  # 목표 공분산행렬을 정의합니다
L = np.linalg.cholesky(cov_matrix)  # 공분산행렬의 촐레스키 인자 L을 구합니다

z = z @ L.T  # 독립 표본에 L의 전치를 곱해 상관 구조를 부여합니다

np.cov(z.T)  # 변환된 표본의 공분산행렬을 계산합니다

array([[0.99424759, 0.80479653, 0.71424917],
       [0.80479653, 1.00962669, 0.81794008],
       [0.71424917, 0.81794008, 1.02650048]])